# Ewaluacja gotowego detektora

Notatnik wczytuje istniejący `model.pkl` i oznaczone dane `test.csv.gz`, wykonuje predykcję i wyświetla podsumowanie kryteriów oraz szczegółowe metryki. **Nie trenuje modelu** i nie potrzebuje `train.csv.gz`.

Umieść plik modelu i test w tym samym katalogu co notatnik albo zmień ścieżki w komórce konfiguracji. Dane wejściowe powinny mieć ten sam schemat przepływów; do wyliczenia czułości/precyzji potrzebna jest kolumna `label`.

## 1. Import bibliotek

In [1]:
from pathlib import Path
import time

import cloudpickle
import numpy as np
import pandas as pd

## 2. Ścieżki do plików

In [2]:
# Domyślnie model.pkl i test.csv.gz leżą obok tego notatnika.
ROOT = Path.cwd()
MODEL_PATH = ROOT / "model.pkl"
TEST_PATH = ROOT / "test.csv.gz"

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Nie znaleziono modelu: {MODEL_PATH}")
if not TEST_PATH.is_file():
    raise FileNotFoundError(f"Nie znaleziono danych testowych: {TEST_PATH}")

## 3. Wczytanie gotowego modelu i testu

In [3]:
# Unpickle wyłącznie modelu z zaufanego źródła; bez uruchamiania fit().
with MODEL_PATH.open("rb") as model_file:
    model = cloudpickle.load(model_file)

test = pd.read_csv(TEST_PATH, keep_default_na=False)
if "label" not in test.columns:
    raise ValueError("Do ewaluacji potrzebna jest kolumna 'label'.")

label_columns = [column for column in ("label", "phase", "raw_label") if column in test.columns]
X_test = test.drop(columns=label_columns)
required_columns = {
    "ts", "duration", "src_ip", "src_port", "dst_ip", "dst_port", "proto",
    "state", "bytes_out", "bytes_in", "pkts_out", "pkts_in",
}
missing = sorted(required_columns.difference(X_test.columns))
if missing:
    raise ValueError(f"Brak wymaganych kolumn: {', '.join(missing)}")

print(f"Model: {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1024**2:.2f} MiB)")
print(f"Dane testowe: {len(test):,} przepływów")

Model: model.pkl (0.63 MiB)
Dane testowe: 515,018 przepływów


## 4. Predykcja i metryki klasyfikacji

In [4]:
# Czas obejmuje predict() oraz całe obliczanie cech modelu.
start = time.perf_counter()
predictions = np.asarray(model.predict(X_test)).astype(bool)
elapsed_seconds = time.perf_counter() - start
labels = test["label"].to_numpy().astype(bool)
if len(predictions) != len(labels):
    raise ValueError("Model zwrócił inną liczbę predykcji niż wierszy danych.")

true_positives = int(np.sum(predictions & labels))
false_positives = int(np.sum(predictions & ~labels))
false_negatives = int(np.sum(~predictions & labels))
true_negatives = int(np.sum(~predictions & ~labels))
total_attacks = int(labels.sum())
total_alarms = int(predictions.sum())
recall = true_positives / max(total_attacks, 1)
precision = true_positives / max(total_alarms, 1)

hours = max((float(test["ts"].max()) - float(test["ts"].min())) / 3600, 1 / 60)
false_alarms_per_hour = false_positives / hours
flows_per_second = len(test) / max(elapsed_seconds, 1e-9)
model_size_mib = MODEL_PATH.stat().st_size / (1024 ** 2)

print(f"Czułość: {recall:.1%} | Precyzja: {precision:.1%}")
print(f"Alarmy: {total_alarms:,} | TP: {true_positives:,} | FP: {false_positives:,} | FN: {false_negatives:,}")

Czułość: 77.5% | Precyzja: 99.5%
Alarmy: 226,337 | TP: 225,134 | FP: 1,203 | FN: 65,317


## 5. Wyniki dla faz ataku

In [5]:
phase_rows = []
if "phase" in test.columns and "network" in test.columns:
    # Incydent = kombinacja sieci i fazy; próg wykrycia zgodny z ewaluatorem modelu.
    false_positive_rate = false_positives / max(int((~labels).sum()), 1)
    for (network, phase), group in test.loc[labels].groupby(["network", "phase"]):
        indexes = group.index.to_numpy()
        phase_recall = float(predictions[indexes].mean())
        phase_rows.append({
            "Sieć": network,
            "Faza": phase,
            "Przepływy ataku": len(group),
            "Alarmy": int(predictions[indexes].sum()),
            "Czułość fazy": phase_recall,
            "Wykryta": bool(predictions[indexes].any() and phase_recall >= 3 * false_positive_rate),
        })

phase_table = pd.DataFrame(phase_rows)
if not phase_table.empty:
    phase_summary = phase_table.groupby("Faza")["Wykryta"].any()
    phase_count = int(len(phase_summary))
    phase_detected = int(phase_summary.sum())
    incidents_count = len(phase_table)
    incidents_detected = int(phase_table["Wykryta"].sum())
else:
    phase_count = phase_detected = incidents_count = incidents_detected = 0

print(f"Wykryte typy faz: {phase_detected}/{phase_count}")
print(f"Wykryte incydenty (sieć × faza): {incidents_detected}/{incidents_count}")

Wykryte typy faz: 5/7
Wykryte incydenty (sieć × faza): 19/30


## 6. Podsumowanie i szczegółowa klasyfikacja

In [6]:
phase_result = (
    f"{phase_detected}/{phase_count} typów; {incidents_detected}/{incidents_count} incydentów"
    if phase_count else "brak danych faz/sieci"
)
budget_status = "spełniony" if false_alarms_per_hour <= 5 else "przekroczony"

criteria_table = pd.DataFrame([
    {"Kryterium": "Rozmiar modelu", "Wynik": f"{model_size_mib:.2f} MiB"},
    {"Kryterium": "Szybkość predykcji", "Wynik": f"{flows_per_second:,.0f} przepływów/s ({elapsed_seconds:.2f} s)"},
    {"Kryterium": "Wykryte fazy przy budżecie 5 FP/h",
     "Wynik": f"{phase_result}; {false_alarms_per_hour:.2f} FP/h ({budget_status})"},
])

classification_table = pd.DataFrame([
    {"Wskaźnik": "Czułość", "Wynik": f"{recall:.1%}"},
    {"Wskaźnik": "Precyzja", "Wynik": f"{precision:.1%}"},
    {"Wskaźnik": "Alarmy ogółem", "Wynik": f"{total_alarms:,}"},
    {"Wskaźnik": "Alarmy trafione (TP)", "Wynik": f"{true_positives:,}"},
    {"Wskaźnik": "Ataki w teście", "Wynik": f"{total_attacks:,}"},
    {"Wskaźnik": "Fałszywe alarmy (FP)", "Wynik": f"{false_positives:,}"},
    {"Wskaźnik": "Ataki niewykryte (FN)", "Wynik": f"{false_negatives:,}"},
    {"Wskaźnik": "Poprawnie odrzucone (TN)", "Wynik": f"{true_negatives:,}"},
])

print("### Trzy kryteria")
display(criteria_table)
print("### Szczegółowe wyniki klasyfikacji")
display(classification_table)
if not phase_table.empty:
    print("### Wyniki według sieci i fazy")
    display(phase_table)

### Trzy kryteria


,Kryterium,Wynik
0,Rozmiar modelu,0.63 MiB
1,Szybkość predykcji,"23,173 przepływów/s (22.22 s)"
2,Wykryte fazy przy budżecie 5 FP/h,5/7 typów; 19/30 incydentów; 50.13 FP/h (przek...


### Szczegółowe wyniki klasyfikacji


,Wskaźnik,Wynik
0,Czułość,77.5%
1,Precyzja,99.5%
2,Alarmy ogółem,"226,337"
3,Alarmy trafione (TP),"225,134"
4,Ataki w teście,"290,451"
5,Fałszywe alarmy (FP),"1,203"
6,Ataki niewykryte (FN),"65,317"
7,Poprawnie odrzucone (TN),"223,364"


### Wyniki według sieci i fazy


,Sieć,Faza,Przepływy ataku,Alarmy,Czułość fazy,Wykryta
0,siec_01,dns_brute_force,1192,1192,1.000000,True
1,siec_01,host_discovery,32852,16907,0.514641,True
2,siec_01,online_cracking,38,1,0.026316,True
3,siec_01,service_scan,9074,9074,1.000000,True
4,siec_01,web_scan,4319,12,0.002778,False
5,siec_01,webshell,15,0,0.000000,False
6,siec_04,dns_brute_force,3239,3239,1.000000,True
7,siec_04,host_discovery,67547,42782,0.633366,True
8,siec_04,service_scan,6800,6800,1.000000,True
9,siec_04,web_scan,9090,268,0.029483,True
